# SHIPP Stage 13 — Marketplace Analytics

**Owner:** Raneem  
**Priority:** P0  
**Current gate:** 13.0 — Analytics CDF source readiness

## Goal

Verify that the two operational Agent-write tables:

- `shipp.saved_items`
- `shipp.agent_activity`

are arriving in Bronze through Lakebase CDF before we build
`gold_marketplace_metrics`.

**This gate is read-only.** It does not create or modify analytics data.

### PASS condition

Both expected Bronze CDF tables must:

1. exist,
2. contain at least one row,
3. expose the SHIPP Lakebase CDF ordering metadata:
   - `_pg_change_type`
   - `_pg_lsn`
   - `_sort_by`


In [0]:
%run ../common/shipp_silver_lib

In [0]:

# STAGE 13 — Gate 13.0A
# Verify canonical Analytics table registration.
# READ ONLY.
# -------------------------------------------------------------------

EXPECTED_SAVED_ITEMS_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_saved_items_history"
)

EXPECTED_AGENT_ACTIVITY_HISTORY = (
    f"{CATALOG}.{BRONZE_SCHEMA}.lb_agent_activity_history"
)

EXPECTED_GOLD_METRICS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_marketplace_metrics"
)


assert BRONZE_SAVED_ITEMS_HISTORY == EXPECTED_SAVED_ITEMS_HISTORY, (
    "CONFIG ERROR: BRONZE_SAVED_ITEMS_HISTORY is incorrect.\n"
    f"Expected: {EXPECTED_SAVED_ITEMS_HISTORY}\n"
    f"Actual:   {BRONZE_SAVED_ITEMS_HISTORY}"
)


assert BRONZE_AGENT_ACTIVITY_HISTORY == EXPECTED_AGENT_ACTIVITY_HISTORY, (
    "CONFIG ERROR: BRONZE_AGENT_ACTIVITY_HISTORY is incorrect.\n"
    f"Expected: {EXPECTED_AGENT_ACTIVITY_HISTORY}\n"
    f"Actual:   {BRONZE_AGENT_ACTIVITY_HISTORY}"
)


assert GOLD_MARKETPLACE_METRICS == EXPECTED_GOLD_METRICS, (
    "CONFIG ERROR: GOLD_MARKETPLACE_METRICS is incorrect.\n"
    f"Expected: {EXPECTED_GOLD_METRICS}\n"
    f"Actual:   {GOLD_MARKETPLACE_METRICS}"
)


print("PASS — Stage 13 canonical table names are registered.")
print()
print("Saved Items CDF :", BRONZE_SAVED_ITEMS_HISTORY)
print("Agent Activity  :", BRONZE_AGENT_ACTIVITY_HISTORY)
print("Gold target     :", GOLD_MARKETPLACE_METRICS)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0B
# Verify real Lakebase CDF sources.
#
# READ ONLY.
#
# Required SHIPP Lakebase CDC metadata:
#   _pg_change_type
#   _pg_lsn
#   _sort_by
# -------------------------------------------------------------------

ANALYTICS_CDF_SOURCES = {
    "saved_items": BRONZE_SAVED_ITEMS_HISTORY,
    "agent_activity": BRONZE_AGENT_ACTIVITY_HISTORY,
}

REQUIRED_CDF_COLUMNS = {
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

NUMERIC_SPARK_TYPES = {
    "byte",
    "short",
    "integer",
    "long",
    "float",
    "double",
    "decimal",
}


print(
    "=== STAGE 13 — ANALYTICS CDF SOURCE READINESS ==="
)


source_results = {}


for source_name, table_name in ANALYTICS_CDF_SOURCES.items():

    print()
    print("--------------------------------------------------")
    print("Source:", source_name)
    print("Table :", table_name)
    print("--------------------------------------------------")

    exists = table_exists(table_name)

    print("Exists:", exists)

    source_results[source_name] = {
        "table": table_name,
        "exists": exists,
        "rows": None,
        "missing_columns": None,
        "change_types": None,
    }


    # ---------------------------------------------------------------
    # Table must actually exist.
    # ---------------------------------------------------------------

    if not exists:

        print(
            "BLOCKED — expected Lakebase CDF "
            "history table does not exist."
        )

        continue


    # ---------------------------------------------------------------
    # Read the real Bronze CDF table.
    # ---------------------------------------------------------------

    df = spark.table(
        table_name
    )

    row_count = df.count()

    print("Rows:", row_count)

    print()
    print("Schema:")

    df.printSchema()


    # ---------------------------------------------------------------
    # Required CDC metadata
    # ---------------------------------------------------------------

    missing_columns = sorted(
        REQUIRED_CDF_COLUMNS
        - set(df.columns)
    )

    source_results[source_name][
        "rows"
    ] = row_count

    source_results[source_name][
        "missing_columns"
    ] = missing_columns


    assert not missing_columns, (
        f"BLOCKED: {table_name} is missing "
        f"required CDF columns: "
        f"{missing_columns}"
    )


    # ---------------------------------------------------------------
    # We need real events, not just an empty table.
    # ---------------------------------------------------------------

    assert row_count > 0, (
        f"BLOCKED: {table_name} exists "
        "but contains 0 rows. "
        "Stage 13 needs real Agent-write "
        "CDF evidence."
    )


    # ---------------------------------------------------------------
    # Validate ordering metadata types.
    #
    # Current-state reconstruction depends on:
    #   _pg_lsn DESC
    #   _sort_by DESC
    # ---------------------------------------------------------------

    lsn_type = (
        df.schema["_pg_lsn"]
        .dataType
    )

    sort_type = (
        df.schema["_sort_by"]
        .dataType
    )


    print()
    print("LSN TYPE  =", lsn_type)
    print("SORT TYPE =", sort_type)


    assert lsn_type.typeName() in NUMERIC_SPARK_TYPES, (
        f"BLOCKED: {table_name}._pg_lsn "
        f"is not numeric: {lsn_type}"
    )


    assert sort_type.typeName() in NUMERIC_SPARK_TYPES, (
        f"BLOCKED: {table_name}._sort_by "
        f"is not numeric: {sort_type}"
    )


    # ---------------------------------------------------------------
    # Show actual change types present.
    # ---------------------------------------------------------------

    change_types = [
        row["_pg_change_type"]
        for row in (
            df
            .select("_pg_change_type")
            .distinct()
            .orderBy("_pg_change_type")
            .collect()
        )
    ]


    source_results[source_name][
        "change_types"
    ] = change_types


    print()
    print(
        "Change types:",
        change_types,
    )


    # ---------------------------------------------------------------
    # Show newest events for runtime evidence.
    # ---------------------------------------------------------------

    print()
    print("Latest CDF rows:")

    display(
        df
        .orderBy(
            F.col("_pg_lsn").desc(),
            F.col("_sort_by").desc(),
        )
        .limit(10)
    )


# -------------------------------------------------------------------
# Final source existence gate.
# -------------------------------------------------------------------

missing_sources = [
    details["table"]
    for details
    in source_results.values()
    if not details["exists"]
]


assert not missing_sources, (
    "STAGE 13 BLOCKED — "
    "missing CDF source(s): "
    + ", ".join(missing_sources)
)


print()
print(
    "PASS Gate 13.0 — saved_items and "
    "agent_activity CDF histories exist, "
    "contain real rows, and expose valid "
    "CDC ordering metadata."
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.0 Summary
# -------------------------------------------------------------------

gate_13_0_summary = {
    "stage": 13,
    "gate": "13.0_cdf_source_readiness",

    "saved_items_history": {
        "table": BRONZE_SAVED_ITEMS_HISTORY,
        "exists": source_results["saved_items"]["exists"],
        "rows": source_results["saved_items"]["rows"],
        "change_types": source_results["saved_items"]["change_types"],
    },

    "agent_activity_history": {
        "table": BRONZE_AGENT_ACTIVITY_HISTORY,
        "exists": source_results["agent_activity"]["exists"],
        "rows": source_results["agent_activity"]["rows"],
        "change_types": source_results["agent_activity"]["change_types"],
    },

    "target": GOLD_MARKETPLACE_METRICS,
    "status": "PASS",
}

print(
    json.dumps(
        gate_13_0_summary,
        indent=2,
        default=str,
    )
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.1
# Reconstruct trusted current state from Analytics CDF histories.
#
# READ ONLY.
#
# CDC rule:
#   1. rank insert / update_postimage / delete
#   2. _pg_lsn DESC
#   3. _sort_by DESC
#   4. select newest event per business key
#   5. newest DELETE = entity excluded from current state
#
# IMPORTANT:
# Never filter DELETE before ranking.
# -------------------------------------------------------------------

print("=== STAGE 13 — TRUSTED ANALYTICS CURRENT STATE ===")


# ================================================================
# Saved Items
# Business key: saved_item_id
# ================================================================

saved_items_bronze = spark.table(
    BRONZE_SAVED_ITEMS_HISTORY
)

(
    saved_items_latest,
    saved_items_current,
    saved_items_deleted,
) = reconstruct_current_state(
    bronze_df=saved_items_bronze,
    key="saved_item_id",
)


saved_items_bronze_count = (
    saved_items_bronze.count()
)

saved_items_latest_count = (
    saved_items_latest.count()
)

saved_items_current_count = (
    saved_items_current.count()
)

saved_items_deleted_count = (
    saved_items_deleted.count()
)

saved_items_duplicate_current = (
    saved_items_current
    .groupBy("saved_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


print()
print("--- SAVED ITEMS ---")
print(
    "Bronze CDF rows       :",
    saved_items_bronze_count,
)
print(
    "Latest business keys  :",
    saved_items_latest_count,
)
print(
    "Current saved items   :",
    saved_items_current_count,
)
print(
    "Latest DELETE keys    :",
    saved_items_deleted_count,
)
print(
    "Duplicate current keys:",
    saved_items_duplicate_current,
)


assert saved_items_duplicate_current == 0, (
    "FAILED: duplicate saved_item_id "
    "after current-state reconstruction."
)


print()
print("Current Saved Items:")

display(
    saved_items_current
    .select(
        "saved_item_id",
        "user_id",
        "request_id",
        "listing_id",
        "status",
        "saved_at",
        "_pg_lsn",
        "_sort_by",
        "_pg_change_type",
    )
    .orderBy("saved_item_id")
)


# ================================================================
# Agent Activity
# Business key: activity_id
# ================================================================

agent_activity_bronze = spark.table(
    BRONZE_AGENT_ACTIVITY_HISTORY
)

(
    agent_activity_latest,
    agent_activity_current,
    agent_activity_deleted,
) = reconstruct_current_state(
    bronze_df=agent_activity_bronze,
    key="activity_id",
)


agent_activity_bronze_count = (
    agent_activity_bronze.count()
)

agent_activity_latest_count = (
    agent_activity_latest.count()
)

agent_activity_current_count = (
    agent_activity_current.count()
)

agent_activity_deleted_count = (
    agent_activity_deleted.count()
)

agent_activity_duplicate_current = (
    agent_activity_current
    .groupBy("activity_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


print()
print("--- AGENT ACTIVITY ---")
print(
    "Bronze CDF rows       :",
    agent_activity_bronze_count,
)
print(
    "Latest business keys  :",
    agent_activity_latest_count,
)
print(
    "Current activity rows :",
    agent_activity_current_count,
)
print(
    "Latest DELETE keys    :",
    agent_activity_deleted_count,
)
print(
    "Duplicate current keys:",
    agent_activity_duplicate_current,
)


assert agent_activity_duplicate_current == 0, (
    "FAILED: duplicate activity_id "
    "after current-state reconstruction."
)


print()
print("Agent Activity by status:")

display(
    agent_activity_current
    .groupBy(
        "tool_name",
        "action_status",
    )
    .count()
    .orderBy(
        "tool_name",
        "action_status",
    )
)


print()
print(
    "PASS Gate 13.1 — trusted Saved Item and "
    "Agent Activity current states reconstructed "
    "with delete-aware CDC semantics."
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.2
# Preview the gold_marketplace_metrics contract.
#
# READ ONLY — this cell does NOT write the Gold table.
#
# Grain:
#   1 row = current SHIPP marketplace analytics snapshot
#
# Business key:
#   metric_key = "marketplace_current"
# -------------------------------------------------------------------

print("=== STAGE 13 — GOLD MARKETPLACE METRICS PREVIEW ===")


# ================================================================
# 1. Validate required Gold matching dependency
# ================================================================

assert table_exists(GOLD_CANDIDATE_MATCHES), (
    f"BLOCKED: required Gold table does not exist: "
    f"{GOLD_CANDIDATE_MATCHES}"
)

candidate_matches_df = spark.table(
    GOLD_CANDIDATE_MATCHES
)

required_gold_columns = {
    "request_id",
    "listing_id",
}

missing_gold_columns = sorted(
    required_gold_columns
    - set(candidate_matches_df.columns)
)

assert not missing_gold_columns, (
    f"BLOCKED: {GOLD_CANDIDATE_MATCHES} "
    f"is missing required columns: "
    f"{missing_gold_columns}"
)


# ================================================================
# 2. Saved Item metrics
# ================================================================

saved_items_count = (
    saved_items_current
    .count()
)

saved_requests_count = (
    saved_items_current
    .select("request_id")
    .distinct()
    .count()
)


# ================================================================
# 3. Agent metrics
#
# Gate 13.1 already reconstructed one current row per activity_id.
# Therefore each row below is one logical Agent action.
# ================================================================

agent_action_count = (
    agent_activity_current
    .count()
)

agent_success_count = (
    agent_activity_current
    .filter(
        F.col("action_status") == "SUCCESS"
    )
    .count()
)

agent_rejected_duplicate_count = (
    agent_activity_current
    .filter(
        F.col("action_status") == "REJECTED_DUPLICATE"
    )
    .count()
)

agent_rejected_unavailable_count = (
    agent_activity_current
    .filter(
        F.col("action_status") == "REJECTED_UNAVAILABLE"
    )
    .count()
)

agent_success_rate = (
    float(agent_success_count)
    / float(agent_action_count)
    if agent_action_count > 0
    else 0.0
)


# ================================================================
# 4. Candidate-match metrics
# ================================================================

candidate_match_count = (
    candidate_matches_df
    .count()
)

requests_with_matches = (
    candidate_matches_df
    .select("request_id")
    .distinct()
    .count()
)


# ================================================================
# 5. Save rate
#
# Current definition:
# requests with >= 1 saved item
# divided by
# requests with >= 1 candidate match
# ================================================================

save_rate = (
    float(saved_requests_count)
    / float(requests_with_matches)
    if requests_with_matches > 0
    else 0.0
)


# ================================================================
# 6. Latest operational event
#
# Use business timestamps from the operational CDF-derived states.
# ================================================================

latest_saved_at = (
    saved_items_current
    .agg(
        F.max("saved_at").alias("latest_saved_at")
    )
    .first()["latest_saved_at"]
)

latest_agent_activity_at = (
    agent_activity_current
    .agg(
        F.max("created_at").alias(
            "latest_agent_activity_at"
        )
    )
    .first()["latest_agent_activity_at"]
)

operational_timestamps = [
    ts
    for ts in [
        latest_saved_at,
        latest_agent_activity_at,
    ]
    if ts is not None
]

latest_operational_event_at = (
    max(operational_timestamps)
    if operational_timestamps
    else None
)


# ================================================================
# 7. Build the one-row Gold preview
# ================================================================

metrics_preview = (
    spark.range(1)
    .select(
        F.lit("marketplace_current")
        .alias("metric_key"),

        F.lit(saved_items_count)
        .cast("long")
        .alias("saved_items_count"),

        F.lit(saved_requests_count)
        .cast("long")
        .alias("saved_requests_count"),

        F.lit(candidate_match_count)
        .cast("long")
        .alias("candidate_match_count"),

        F.lit(requests_with_matches)
        .cast("long")
        .alias("requests_with_matches"),

        F.lit(save_rate)
        .cast("double")
        .alias("save_rate"),

        F.lit(agent_action_count)
        .cast("long")
        .alias("agent_action_count"),

        F.lit(agent_success_count)
        .cast("long")
        .alias("agent_success_count"),

        F.lit(agent_rejected_duplicate_count)
        .cast("long")
        .alias("agent_rejected_duplicate_count"),

        F.lit(agent_rejected_unavailable_count)
        .cast("long")
        .alias("agent_rejected_unavailable_count"),

        F.lit(agent_success_rate)
        .cast("double")
        .alias("agent_success_rate"),

        F.lit(latest_operational_event_at)
        .cast("timestamp_ntz")
        .alias("latest_operational_event_at"),

        F.current_timestamp()
        .alias("analytics_processed_at"),
    )
)


# ================================================================
# 8. Exact schema contract
# ================================================================

EXPECTED_METRICS_COLUMNS = [
    "metric_key",
    "saved_items_count",
    "saved_requests_count",
    "candidate_match_count",
    "requests_with_matches",
    "save_rate",
    "agent_action_count",
    "agent_success_count",
    "agent_rejected_duplicate_count",
    "agent_rejected_unavailable_count",
    "agent_success_rate",
    "latest_operational_event_at",
    "analytics_processed_at",
]

assert metrics_preview.columns == EXPECTED_METRICS_COLUMNS, (
    "FAILED: metrics preview schema does not "
    "match the Stage 13 contract."
)


# ================================================================
# 9. Quality gates
# ================================================================

preview_count = metrics_preview.count()

assert preview_count == 1, (
    f"FAILED: expected exactly 1 metrics row, "
    f"found {preview_count}"
)

assert (
    metrics_preview
    .filter(
        F.col("metric_key").isNull()
    )
    .count()
    == 0
), "FAILED: metric_key is NULL."

assert (
    metrics_preview
    .filter(
        (F.col("saved_items_count") < 0)
        | (F.col("saved_requests_count") < 0)
        | (F.col("candidate_match_count") < 0)
        | (F.col("requests_with_matches") < 0)
        | (F.col("agent_action_count") < 0)
        | (F.col("agent_success_count") < 0)
    )
    .count()
    == 0
), "FAILED: negative metric count detected."

assert 0.0 <= save_rate <= 1.0, (
    f"FAILED: save_rate outside 0..1: "
    f"{save_rate}"
)

assert 0.0 <= agent_success_rate <= 1.0, (
    f"FAILED: agent_success_rate outside 0..1: "
    f"{agent_success_rate}"
)

assert agent_success_count <= agent_action_count, (
    "FAILED: Agent successes exceed total Agent actions."
)


# ================================================================
# 10. Runtime evidence
# ================================================================

print()
print("Saved Items count              :", saved_items_count)
print("Saved Requests count           :", saved_requests_count)

print()
print("Candidate Match count           :", candidate_match_count)
print("Requests with matches           :", requests_with_matches)
print("Save rate                       :", save_rate)

print()
print("Agent action count              :", agent_action_count)
print("Agent success count             :", agent_success_count)
print(
    "Rejected duplicate count       :",
    agent_rejected_duplicate_count,
)
print(
    "Rejected unavailable count     :",
    agent_rejected_unavailable_count,
)
print("Agent success rate              :", agent_success_rate)

print()
print(
    "Latest operational event at    :",
    latest_operational_event_at,
)

print()
print("Gold metrics preview:")

display(metrics_preview)


print()
print(
    "PASS Gate 13.2 — "
    "gold_marketplace_metrics contract preview is valid."
)

In [0]:
# -------------------------------------------------------------------
# STAGE 13 — Gate 13.3
# Persist gold_marketplace_metrics.
#
# TARGET:
#   bootcamp_students.shipp_gold.gold_marketplace_metrics
#
# GRAIN:
#   exactly one row per metric_key
#
# BUSINESS KEY:
#   metric_key
#
# WRITE SEMANTICS:
#   MERGE / UPSERT
#
# IMPORTANT:
#   Rerunning this cell must UPDATE marketplace_current,
#   never append a duplicate marketplace_current row.
# -------------------------------------------------------------------

print("=== STAGE 13 — PERSIST GOLD MARKETPLACE METRICS ===")

TARGET_TABLE = GOLD_MARKETPLACE_METRICS

print("Target:", TARGET_TABLE)


# ================================================================
# 1. Ensure Gold schema exists
# ================================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)


# ================================================================
# 2. Persist using metric_key as the logical business key
# ================================================================

target_existed_before = table_exists(
    TARGET_TABLE
)

print(
    "Target existed before write:",
    target_existed_before,
)


if not target_existed_before:

    (
        metrics_preview
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

    print(
        "Created Gold metrics table."
    )

else:

    metrics_preview.createOrReplaceTempView(
        "stage13_metrics_preview"
    )

    spark.sql(
        f"""
        MERGE INTO {TARGET_TABLE} AS target
        USING stage13_metrics_preview AS source

        ON target.metric_key = source.metric_key

        WHEN MATCHED THEN
          UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *
        """
    )

    print(
        "Merged marketplace_current "
        "into existing Gold metrics table."
    )


# ================================================================
# 3. Read persisted Gold product
# ================================================================

gold_metrics = spark.table(
    TARGET_TABLE
)

persisted_row_count = (
    gold_metrics.count()
)

duplicate_metric_keys = (
    gold_metrics
    .groupBy("metric_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print()
print(
    "Persisted row count     :",
    persisted_row_count,
)

print(
    "Duplicate metric keys   :",
    duplicate_metric_keys,
)


# ================================================================
# 4. Exact schema contract
# ================================================================

assert gold_metrics.columns == EXPECTED_METRICS_COLUMNS, (
    "FAILED: persisted Gold metrics schema "
    "does not match Gate 13.2 contract.\n"
    f"Expected: {EXPECTED_METRICS_COLUMNS}\n"
    f"Actual:   {gold_metrics.columns}"
)


# ================================================================
# 5. Grain / business-key validation
# ================================================================

assert persisted_row_count == 1, (
    "FAILED: gold_marketplace_metrics must "
    f"contain exactly 1 current metrics row; "
    f"found {persisted_row_count}."
)

assert duplicate_metric_keys == 0, (
    "FAILED: duplicate metric_key detected."
)

assert (
    gold_metrics
    .filter(
        F.col("metric_key")
        == "marketplace_current"
    )
    .count()
    == 1
), (
    "FAILED: marketplace_current row "
    "is missing."
)


# ================================================================
# 6. Validate persisted metric values
# ================================================================

persisted = (
    gold_metrics
    .filter(
        F.col("metric_key")
        == "marketplace_current"
    )
    .first()
)


assert persisted["saved_items_count"] == saved_items_count
assert persisted["saved_requests_count"] == saved_requests_count

assert (
    persisted["candidate_match_count"]
    == candidate_match_count
)

assert (
    persisted["requests_with_matches"]
    == requests_with_matches
)

assert abs(
    persisted["save_rate"]
    - save_rate
) < 1e-12

assert (
    persisted["agent_action_count"]
    == agent_action_count
)

assert (
    persisted["agent_success_count"]
    == agent_success_count
)

assert (
    persisted[
        "agent_rejected_duplicate_count"
    ]
    == agent_rejected_duplicate_count
)

assert (
    persisted[
        "agent_rejected_unavailable_count"
    ]
    == agent_rejected_unavailable_count
)

assert abs(
    persisted["agent_success_rate"]
    - agent_success_rate
) < 1e-12


# ================================================================
# 7. Runtime evidence
# ================================================================

print()
print("Persisted Gold marketplace metrics:")

display(
    gold_metrics
    .orderBy("metric_key")
)


print()
print(
    "PASS Gate 13.3 — "
    "gold_marketplace_metrics persisted "
    "with one unique marketplace_current row."
)

In [0]:
# ============================================================
# STAGE 13 — FINAL PRODUCTION VALIDATION
# Generic validation safe for pipeline reruns
# ============================================================

print("=== STAGE 13 — FINAL PRODUCTION VALIDATION ===")

# Current Saved Items reconstructed earlier in the notebook.
bronze_saved_count = saved_items_current.count()

# Persisted Gold metrics.
gold_rows = spark.sql("""
SELECT
    metric_key,
    saved_items_count,
    agent_action_count,
    agent_success_count,
    agent_rejected_duplicate_count,
    agent_rejected_unavailable_count,
    agent_success_rate,
    latest_operational_event_at
FROM bootcamp_students.shipp_gold.gold_marketplace_metrics
WHERE metric_key = 'marketplace_current'
""").collect()

assert len(gold_rows) == 1, (
    f"Expected exactly one marketplace_current row, "
    f"found {len(gold_rows)}"
)

gold = gold_rows[0]

print("Bronze current saved_items =", bronze_saved_count)
print("Gold saved_items_count     =", gold["saved_items_count"])
print("Agent action count         =", gold["agent_action_count"])
print("Agent success count        =", gold["agent_success_count"])
print("Latest operational event   =", gold["latest_operational_event_at"])

# Gold must agree with trusted reconstructed CDC state.
assert gold["saved_items_count"] == bronze_saved_count, (
    f"Analytics mismatch: Bronze current saved_items="
    f"{bronze_saved_count}, Gold saved_items_count="
    f"{gold['saved_items_count']}"
)

# Metric key must remain unique.
duplicate_keys = spark.sql("""
SELECT metric_key, COUNT(*) AS row_count
FROM bootcamp_students.shipp_gold.gold_marketplace_metrics
GROUP BY metric_key
HAVING COUNT(*) > 1
""").count()

print("Duplicate metric keys      =", duplicate_keys)

assert duplicate_keys == 0, (
    f"Found {duplicate_keys} duplicate metric keys"
)

print("\nPASS — STAGE 13 FINAL PRODUCTION VALIDATION")
print("Bronze and Gold saved-item counts agree.")
print("Gold metric key remains unique.")